# SwitchLingua Update 2 — Model Training (Person 3)

Trains the causal dual-head model on a Colab GPU and produces:

- the learning-rate choice for XLM-R (validation F1),
- the **XLM-R vs mBERT** backbone comparison (Approach A),
- the **Approach A vs C** punctuation comparison (XLM-R),
- `results/training_comparison.csv` and `docs/figures/training_convergence.png`.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Get the code and install dependencies

In [ ]:
import os
REPO = "/content/SwitchLingua-Project-CSTeam-1"
if not os.path.exists(REPO):
    !git clone -q https://github.com/Chuweic/SwitchLingua-Project-CSTeam-1.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt gdown
!git log --oneline -1

## 3. Download the processed data

Downloads `train.parquet`, `validation.parquet`, and `test.parquet` from the team's shared Drive folder.
If this fails, upload the three files to `data/processed/` with the file browser on the left.

In [ ]:
import os
import pandas as pd

DATA_DIR = "data/processed"
os.makedirs(DATA_DIR, exist_ok=True)
if not all(os.path.exists(f"{DATA_DIR}/{split}.parquet") for split in ("train", "validation", "test")):
    !gdown -q --folder "https://drive.google.com/drive/folders/1W4y0YEtRhuCIa31Y591JgslPQya9AR5o" -O {DATA_DIR}

expected = {"train": 186480, "validation": 23310, "test": 23310}
for split, rows in expected.items():
    found = len(pd.read_parquet(f"{DATA_DIR}/{split}.parquet", columns=["row_id"]))
    assert found == rows, f"{split}.parquet has {found} rows, expected {rows}"
    print(split, found)

## 4. Run the unit tests

In [ ]:
!PYTHONPATH=. python -m pytest -q 2>&1 | tail -3

## 5. Learning-rate search (XLM-R, Approach A)

Three runs that differ only in the learning rate. The best one is chosen by **validation** Anticipatory F1
and becomes the XLM-R / Approach A result, so the test set is not used for tuning.

In [ ]:
LEARNING_RATES = ["2e-5", "3e-5", "5e-5"]
EPOCHS = 3

for lr in LEARNING_RATES:
    output = f"results/tune_xlmr_keep_lr{lr}.json"
    if os.path.exists(output):
        print("skip", output)
        continue
    !PYTHONPATH=. python -m src.train --backbone xlmr --punctuation-mode keep --cross-eval-mode neutral \
        --learning-rate {lr} --epochs {EPOCHS} --output {output} 2>&1 | grep -E "^epoch [0-9]+ done|^TEST|Error|Traceback"

In [ ]:
import json
import shutil

tuning = []
for lr in LEARNING_RATES:
    run = json.load(open(f"results/tune_xlmr_keep_lr{lr}.json"))
    tuning.append({
        "learning_rate": lr,
        "best_epoch": run["history"]["best_epoch"],
        "validation_f1": run["history"]["best_validation_anticipatory_f1"],
        "test_f1": run["overall"]["anticipatory_f1"],
    })
tuning = pd.DataFrame(tuning)
tuning.to_csv("results/learning_rate_search.csv", index=False)

BEST_LR = tuning.sort_values("validation_f1", ascending=False).iloc[0]["learning_rate"]
shutil.copy(f"results/tune_xlmr_keep_lr{BEST_LR}.json", "results/train_xlmr_keep.json")
print("Selected learning rate:", BEST_LR)
tuning

## 6. Backbone comparison and A/C comparison

Same seed, rows, epochs, and learning rate as the selected XLM-R / A run:

- **mBERT, Approach A** — backbone comparison
- **XLM-R, Approach C** — punctuation comparison (also scored on A's labels)
- **mBERT, Approach C** — completes the 2×2 table

In [ ]:
RUNS = [
    ("mbert", "keep", "neutral"),
    ("xlmr", "neutral", "keep"),
    ("mbert", "neutral", "keep"),
]
for backbone, mode, cross in RUNS:
    output = f"results/train_{backbone}_{mode}.json"
    if os.path.exists(output):
        print("skip", output)
        continue
    print(f"=== {backbone} / {mode} ===")
    !PYTHONPATH=. python -m src.train --backbone {backbone} --punctuation-mode {mode} --cross-eval-mode {cross} \
        --learning-rate {BEST_LR} --epochs {EPOCHS} --output {output} 2>&1 | grep -E "^epoch [0-9]+ done|^TEST|Error|Traceback"

## 7. Comparison table and convergence figure

In [ ]:
!PYTHONPATH=. python examples/compare_training_runs.py > /dev/null
comparison = pd.read_csv("results/training_comparison.csv")
comparison.drop(columns=["File"]).round(4)

In [ ]:
from IPython.display import Image
Image("docs/figures/training_convergence.png", width=1000)

### A vs C on the same labels

A and C use different valid-target sets, so their own-label F1 values are not measured on the same positions.
This table scores each XLM-R model on both label sets.

In [ ]:
rows = []
for mode in ("keep", "neutral"):
    run = json.load(open(f"results/train_xlmr_{mode}.json"))
    cross = run["cross_eval"]
    scores = {mode: run["overall"], cross["label_punctuation_mode"]: cross["overall"]}
    rows.append({
        "Trained with": {"keep": "A (keep)", "neutral": "C (neutral)"}[mode],
        "F1 on A labels": scores["keep"]["anticipatory_f1"],
        "F1 on C labels": scores["neutral"]["anticipatory_f1"],
        "Duration acc on A labels": scores["keep"]["duration_accuracy"],
        "Duration acc on C labels": scores["neutral"]["duration_accuracy"],
        "Universality σ (own labels)": run["universality_std"],
    })
ac = pd.DataFrame(rows)
ac.to_csv("results/punctuation_ac_model_comparison.csv", index=False)
ac.round(4)

## 8. Download the results

In [ ]:
!zip -q -r /content/update2_training_results.zip results/train_*.json results/tune_*.json \
    results/learning_rate_search.csv results/training_comparison.csv \
    results/punctuation_ac_model_comparison.csv docs/figures/training_convergence.png
from google.colab import files
files.download("/content/update2_training_results.zip")